# MNQ Pullback retenu - Notebook final client

Ce notebook reconstruit la sleeve `volume climax pullback` retenue dans l'etude MNQ ORB + Pullback.

Important :
- le code de la strategie est visible dans le notebook,
- l'alpha et le sizing sont dissocies,
- les heatmaps IS/OOS couvrent le signal, les modes d'entree, les sorties et le sizing,
- le parametrage par defaut reproduit la sleeve utilisee dans le notebook equal-weight client.

In [ ]:
import json
import math
import datetime as dt
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Impossible de retrouver la racine du repo depuis le notebook.")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 240)


In [ ]:
REPO_ROOT = ROOT
DEFAULT_TIMEZONE = "America/New_York"
RTH_START = "09:30"
RTH_END = "16:00"
ETH_START = "18:00"
ETH_END = "17:00"

INSTRUMENT_SPECS = {
    "NQ": {"tick_size": 0.25, "tick_value_usd": 5.0, "point_value_usd": 20.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "MNQ": {"tick_size": 0.25, "tick_value_usd": 0.5, "point_value_usd": 2.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "ES": {"tick_size": 0.25, "tick_value_usd": 12.5, "point_value_usd": 50.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "MES": {"tick_size": 0.25, "tick_value_usd": 1.25, "point_value_usd": 5.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "RTY": {"tick_size": 0.10, "tick_value_usd": 5.0, "point_value_usd": 50.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "M2K": {"tick_size": 0.10, "tick_value_usd": 0.5, "point_value_usd": 5.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "MGC": {"tick_size": 0.10, "tick_value_usd": 1.0, "point_value_usd": 10.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
}

DEFAULT_SYMBOL = "MNQ"
DEFAULT_TICK_SIZE = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["tick_size"]
DEFAULT_TICK_VALUE_USD = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["tick_value_usd"]
DEFAULT_POINT_VALUE_USD = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["point_value_usd"]
DEFAULT_COMMISSION_PER_SIDE_USD = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["commission_per_side_usd"]
DEFAULT_SLIPPAGE_TICKS = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["slippage_ticks"]
DEFAULT_INITIAL_CAPITAL_USD = 50_000.0
DEFAULT_MONTHLY_SUBSCRIPTION_COST_USD = 150.0

def get_instrument_spec(symbol: str) -> dict:
    key = symbol.upper()
    if key not in INSTRUMENT_SPECS:
        raise ValueError(f"Unknown instrument {symbol!r}")
    return INSTRUMENT_SPECS[key].copy()

def resolve_dataset_path(explicit_path, symbol: str, timeframe: str = "1m") -> Path:
    if explicit_path is not None:
        return Path(explicit_path)
    files = sorted((REPO_ROOT / "data" / "processed" / "parquet").glob(f"{symbol}_c_0_{timeframe}_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No processed dataset found for {symbol} {timeframe}.")
    return files[-1]

def fmt_money(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.1f} USD"

def fmt_pct(value, digits=2):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}%"

def fmt_float(value, digits=3):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"

def parameter_table(rows):
    frame = pd.DataFrame(rows)
    display(frame)
    return frame

def normalize_sessions(sessions):
    return pd.to_datetime(pd.Index(sessions), errors="coerce").normalize().tolist()

def session_set(sessions):
    return set(pd.to_datetime(pd.Index(sessions), errors="coerce").date)

def subset_trades(trades, sessions):
    if trades.empty:
        return trades.copy()
    allowed = session_set(sessions)
    out = trades.copy()
    out["_session_key"] = pd.to_datetime(out["session_date"], errors="coerce").dt.date
    return out.loc[out["_session_key"].isin(allowed)].drop(columns=["_session_key"]).copy().reset_index(drop=True)

def daily_results_from_trades(trades, sessions, initial_capital):
    calendar = pd.DataFrame({"session_date": normalize_sessions(sessions)})
    calendar = calendar.dropna().drop_duplicates().sort_values("session_date").reset_index(drop=True)
    if calendar.empty:
        return pd.DataFrame(columns=["session_date", "daily_pnl_usd", "daily_trade_count", "equity", "peak_equity", "drawdown_usd", "drawdown_pct", "daily_return"])

    if trades.empty:
        daily = calendar.copy()
        daily["daily_pnl_usd"] = 0.0
        daily["daily_trade_count"] = 0
    else:
        view = trades.copy()
        view["session_date"] = pd.to_datetime(view["session_date"], errors="coerce").dt.normalize()
        grouped = (
            view.groupby("session_date", as_index=False)
            .agg(daily_pnl_usd=("net_pnl_usd", "sum"), daily_trade_count=("trade_id", "count"))
        )
        daily = calendar.merge(grouped, on="session_date", how="left")
        daily["daily_pnl_usd"] = pd.to_numeric(daily["daily_pnl_usd"], errors="coerce").fillna(0.0)
        daily["daily_trade_count"] = pd.to_numeric(daily["daily_trade_count"], errors="coerce").fillna(0).astype(int)

    prev_equity = float(initial_capital)
    rets = []
    equities = []
    for pnl in pd.to_numeric(daily["daily_pnl_usd"], errors="coerce").fillna(0.0):
        rets.append(float(pnl) / prev_equity if prev_equity else 0.0)
        prev_equity += float(pnl)
        equities.append(prev_equity)

    daily["daily_return"] = rets
    daily["equity"] = equities
    daily["peak_equity"] = daily["equity"].cummax()
    daily["drawdown_usd"] = daily["equity"] - daily["peak_equity"]
    daily["drawdown_pct"] = np.where(daily["peak_equity"] > 0, (daily["equity"] / daily["peak_equity"] - 1.0) * 100.0, 0.0)
    return daily

def curve_from_returns(session_dates, daily_return, initial_capital, label="curve"):
    out = pd.DataFrame({"session_date": pd.to_datetime(pd.Series(session_dates), errors="coerce").dt.normalize()})
    out["daily_return"] = pd.to_numeric(pd.Series(daily_return), errors="coerce").fillna(0.0)
    if (out["daily_return"] <= -1.0).any():
        worst = float(out["daily_return"].min())
        raise ValueError(f"{label}: daily return <= -100% ({worst:.2%}).")
    out["equity"] = float(initial_capital) * (1.0 + out["daily_return"]).cumprod()
    out["daily_pnl_usd"] = out["equity"].diff().fillna(out["equity"].iloc[0] - float(initial_capital))
    out["peak_equity"] = out["equity"].cummax()
    out["drawdown_usd"] = out["equity"] - out["peak_equity"]
    out["drawdown_pct"] = np.where(out["peak_equity"] > 0, (out["equity"] / out["peak_equity"] - 1.0) * 100.0, 0.0)
    return out

def curve_metrics(curve, trades=None, initial_capital=50_000.0):
    if curve.empty:
        return {
            "net_pnl_usd": 0.0,
            "return_pct": 0.0,
            "cagr_pct": np.nan,
            "sharpe": 0.0,
            "sortino": 0.0,
            "annualized_vol_pct": 0.0,
            "profit_factor": 0.0,
            "max_drawdown_usd": 0.0,
            "max_drawdown_pct": 0.0,
            "worst_day_usd": 0.0,
            "n_trades": 0,
            "win_rate": 0.0,
        }
    ordered = curve.sort_values("session_date").reset_index(drop=True)
    rets = pd.to_numeric(ordered["daily_return"], errors="coerce").fillna(0.0)
    equity = pd.to_numeric(ordered["equity"], errors="coerce").fillna(float(initial_capital))
    pnl = pd.to_numeric(ordered["daily_pnl_usd"], errors="coerce").fillna(0.0)
    start = pd.Timestamp(ordered["session_date"].iloc[0])
    end = pd.Timestamp(ordered["session_date"].iloc[-1])
    years = max(((end - start).days + 1) / 365.25, 1 / 365.25)
    final_equity = float(equity.iloc[-1])
    sharpe = float(rets.mean() / rets.std(ddof=0) * math.sqrt(252.0)) if len(rets) > 1 and rets.std(ddof=0) > 0 else 0.0
    downside = rets[rets < 0]
    downside_std = float(np.sqrt(np.mean(np.square(downside)))) if len(downside) else 0.0
    sortino = float(rets.mean() / downside_std * math.sqrt(252.0)) if downside_std > 0 else 0.0
    cagr = float(((final_equity / float(initial_capital)) ** (1.0 / years) - 1.0) * 100.0) if final_equity > 0 else np.nan
    if trades is not None and not trades.empty:
        trade_pnl = pd.to_numeric(trades["net_pnl_usd"], errors="coerce").fillna(0.0)
        wins = trade_pnl[trade_pnl > 0]
        losses = trade_pnl[trade_pnl < 0]
        profit_factor = float(wins.sum() / abs(losses.sum())) if abs(losses.sum()) > 0 else float("inf")
        n_trades = int(len(trade_pnl))
        win_rate = float((trade_pnl > 0).mean())
    else:
        wins = pnl[pnl > 0]
        losses = pnl[pnl < 0]
        profit_factor = float(wins.sum() / abs(losses.sum())) if abs(losses.sum()) > 0 else float("inf")
        n_trades = 0
        win_rate = np.nan
    return {
        "net_pnl_usd": float(final_equity - float(initial_capital)),
        "return_pct": float((final_equity / float(initial_capital) - 1.0) * 100.0),
        "cagr_pct": cagr,
        "sharpe": sharpe,
        "sortino": sortino,
        "annualized_vol_pct": float(rets.std(ddof=0) * math.sqrt(252.0) * 100.0) if len(rets) > 1 else 0.0,
        "profit_factor": profit_factor,
        "max_drawdown_usd": float(abs(pd.to_numeric(ordered["drawdown_usd"], errors="coerce").min())),
        "max_drawdown_pct": float(abs(pd.to_numeric(ordered["drawdown_pct"], errors="coerce").min())),
        "worst_day_usd": float(pnl.min()),
        "n_trades": n_trades,
        "win_rate": win_rate,
    }

def summarize_strategy_scopes(label, full_curve, full_trades, is_sessions, oos_sessions, initial_capital):
    is_curve = full_curve.loc[full_curve["session_date"].isin(set(normalize_sessions(is_sessions)))].copy()
    oos_trades = subset_trades(full_trades, oos_sessions)
    oos_curve = daily_results_from_trades(oos_trades, oos_sessions, initial_capital)
    rows = [
        {"strategy": label, "scope": "full", **curve_metrics(full_curve, full_trades, initial_capital)},
        {"strategy": label, "scope": "is", **curve_metrics(is_curve, subset_trades(full_trades, is_sessions), initial_capital)},
        {"strategy": label, "scope": "oos", **curve_metrics(oos_curve, oos_trades, initial_capital)},
    ]
    return pd.DataFrame(rows)

def _metric_columns_for_scope(frame, scope, metric):
    mapping = {
        "sharpe": f"{scope}_sharpe_ratio",
        "net_pnl_usd": f"{scope}_net_pnl",
        "profit_factor": f"{scope}_profit_factor",
        "max_drawdown_usd": f"{scope}_max_drawdown",
        "prop_score": f"{scope}_prop_score",
        "n_trades": f"{scope}_nb_trades",
    }
    return mapping[metric]

def plot_scope_heatmap(frame, x, metric, title, text_auto=".2f", color_continuous_scale="RdYlGn"):
    data = frame.copy()
    data[x] = data[x].astype(str)
    pivot = data.pivot_table(index="scope", columns=x, values=metric, aggfunc="mean")
    fig = px.imshow(pivot, aspect="auto", text_auto=text_auto, color_continuous_scale=color_continuous_scale, title=title)
    fig.update_layout(template=PLOT_TEMPLATE, width=1100, height=380)
    fig.update_xaxes(title=x)
    fig.update_yaxes(title="scope")
    fig.show()

def plot_is_oos_heatmaps(frame, x, y, metric, title, text_auto=".2f", color_continuous_scale="RdYlGn"):
    scopes = [("is", "IS"), ("oos", "OOS")]
    fig = make_subplots(rows=1, cols=2, subplot_titles=[label for _, label in scopes], horizontal_spacing=0.10)
    zmin = None
    zmax = None
    if metric not in {"max_drawdown_usd"}:
        finite = pd.to_numeric(frame[metric], errors="coerce")
        if finite.notna().any():
            zmin = float(finite.min())
            zmax = float(finite.max())
    for idx, (scope, label) in enumerate(scopes, start=1):
        scoped = frame.loc[frame["scope"].eq(scope)].copy()
        scoped[x] = scoped[x].astype(str)
        scoped[y] = scoped[y].astype(str)
        pivot = scoped.pivot_table(index=y, columns=x, values=metric, aggfunc="mean").sort_index()
        heatmap = go.Heatmap(
            z=pivot.values,
            x=[str(v) for v in pivot.columns],
            y=[str(v) for v in pivot.index],
            colorscale=color_continuous_scale,
            zmin=zmin,
            zmax=zmax,
            colorbar=dict(title=metric) if idx == 2 else None,
            showscale=idx == 2,
            text=np.round(pivot.values, 2 if metric not in {"net_pnl_usd", "max_drawdown_usd"} else 0),
            texttemplate="%{text}",
        )
        fig.add_trace(heatmap, row=1, col=idx)
    fig.update_layout(template=PLOT_TEMPLATE, width=1350, height=500, title=title)
    fig.update_xaxes(title=x, row=1, col=1)
    fig.update_xaxes(title=x, row=1, col=2)
    fig.update_yaxes(title=y, row=1, col=1)
    fig.update_yaxes(title=y, row=1, col=2)
    fig.show()

def row_to_scope_records(row, extra=None):
    extra = extra or {}
    records = []
    for scope in ("is", "oos"):
        records.append(
            {
                **extra,
                "scope": scope,
                "net_pnl_usd": float(row.get(f"{scope}_net_pnl", 0.0)),
                "sharpe": float(row.get(f"{scope}_sharpe_ratio", 0.0)),
                "profit_factor": float(row.get(f"{scope}_profit_factor", 0.0)),
                "max_drawdown_usd": abs(float(row.get(f"{scope}_max_drawdown", 0.0))),
                "prop_score": float(row.get(f"{scope}_prop_score", 0.0)),
                "n_trades": int(row.get(f"{scope}_nb_trades", 0) or 0),
            }
        )
    return records

def clean_ohlcv(df: pd.DataFrame, drop_duplicate_timestamps: bool = True) -> pd.DataFrame:
    """Sort, type-cast numeric columns, and optionally remove duplicate timestamps."""
    out = df.copy()
    out = out.sort_values("timestamp").reset_index(drop=True)

    if drop_duplicate_timestamps:
        out = out.drop_duplicates(subset=["timestamp"], keep="last")

    numeric_columns = [c for c in ["open", "high", "low", "close", "volume", "open interest"] if c in out.columns]
    for col in numeric_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=["timestamp", "open", "high", "low", "close", "volume"])
    return out.reset_index(drop=True)

def _normalize_ohlcv_frame(df: pd.DataFrame, timezone: str) -> pd.DataFrame:
    """Normalize columns and timestamps on a dataframe already loaded from disk."""
    out = df.copy()
    out.columns = [col.strip().lower() for col in out.columns]

    if "timestamp" not in out.columns and "ts_event" not in out.columns:
        raise ValueError("Missing required 'timestamp' or 'ts_event' column")

    timestamp_col = "timestamp" if "timestamp" in out.columns else "ts_event"
    out[timestamp_col] = pd.to_datetime(out[timestamp_col], errors="coerce")
    if out[timestamp_col].isna().any():
        raise ValueError(f"Found unparsable {timestamp_col} values")

    if timestamp_col != "timestamp":
        out = out.rename(columns={timestamp_col: "timestamp"})

    if out["timestamp"].dt.tz is None:
        out["timestamp"] = out["timestamp"].dt.tz_localize(timezone)
    else:
        out["timestamp"] = out["timestamp"].dt.tz_convert(timezone)

    return out

def load_ohlcv_csv(path: Path | str, timezone: str = DEFAULT_TIMEZONE) -> pd.DataFrame:
    """Load OHLCV csv with timestamp parsing and normalized lowercase columns.

    Assumptions:
    - timestamps in source csv are US Eastern local times
    - timestamp marks beginning of bar
    - intraday bars may have missing minutes due to zero-volume bars not present
    """
    csv_path = Path(path)
    return _normalize_ohlcv_frame(pd.read_csv(csv_path), timezone=timezone)

def load_ohlcv_file(path: Path | str, timezone: str = DEFAULT_TIMEZONE) -> pd.DataFrame:
    """Load a supported OHLCV file and normalize timestamps/column names."""
    file_path = Path(path)
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return load_ohlcv_csv(file_path, timezone=timezone)
    if suffix == ".parquet":
        df = pd.read_parquet(file_path)
        if df.index.name == 'timestamp' or 'timestamp' not in df.columns:
            df = df.reset_index()
        return _normalize_ohlcv_frame(df, timezone=timezone)

    raise ValueError(f"Unsupported OHLCV file format: {file_path.suffix}")

def _session_mask(timestamps: pd.Series, start_time: str, end_time: str) -> pd.Series:
    """Build boolean mask for a time range, handling overnight windows."""
    times = timestamps.dt.time
    start = dt.time.fromisoformat(start_time)
    end = dt.time.fromisoformat(end_time)
    if start <= end:
        return (times >= start) & (times <= end)
    return (times >= start) | (times <= end)

def filter_session(df: pd.DataFrame, start_time: str, end_time: str) -> pd.DataFrame:
    """Filter bars between start_time and end_time (inclusive), local dataframe timezone."""
    mask = _session_mask(df["timestamp"], start_time, end_time)
    return df.loc[mask].reset_index(drop=True)

def extract_rth(df: pd.DataFrame) -> pd.DataFrame:
    """Extract regular trading hours."""
    return filter_session(df, RTH_START, RTH_END)

def add_session_date(df: pd.DataFrame) -> pd.DataFrame:
    """Add session_date from local timestamp date for robust session grouping."""
    out = df.copy()
    out["session_date"] = out["timestamp"].dt.date
    return out

def build_session_time(session_timestamp: pd.Timestamp, clock_time: str) -> pd.Timestamp:
    """Return the session date at the requested clock time, preserving timezone."""
    time_value = dt.time.fromisoformat(clock_time)
    timestamp = pd.Timestamp(session_timestamp)
    return timestamp.replace(
        hour=time_value.hour,
        minute=time_value.minute,
        second=time_value.second,
        microsecond=0,
        nanosecond=0,
    )

def add_intraday_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add minute-of-day, weekday, and bar rank in session."""
    out = df.copy()
    out["session_date"] = out["timestamp"].dt.date
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["weekday"] = out["timestamp"].dt.dayofweek
    out["bar_in_session"] = out.groupby("session_date").cumcount() + 1
    return out

def add_session_vwap(
    df: pd.DataFrame,
    price_mode: str = "typical",
    price_volume_col: str | None = None,
) -> pd.DataFrame:
    """Add an intraday VWAP column that resets each session."""
    out = df.copy()
    if "session_date" not in out.columns:
        out["session_date"] = out["timestamp"].dt.date

    volume = out["volume"].fillna(0.0)

    if price_volume_col is not None:
        if price_volume_col not in out.columns:
            raise ValueError(f"Missing precomputed price-volume column '{price_volume_col}'.")
        pv = pd.to_numeric(out[price_volume_col], errors="coerce").fillna(0.0)
    elif price_mode == "close":
        price = out["close"]
        pv = price * volume
    elif price_mode == "typical":
        price = (out["high"] + out["low"] + out["close"]) / 3.0
        pv = price * volume
    else:
        raise ValueError("price_mode must be 'close' or 'typical'.")

    cumulative_pv = pv.groupby(out["session_date"]).cumsum()
    cumulative_volume = volume.groupby(out["session_date"]).cumsum()
    out["session_vwap"] = np.where(cumulative_volume > 0, cumulative_pv / cumulative_volume, np.nan)
    return out

def add_continuous_session_vwap(
    df: pd.DataFrame,
    price_mode: str = "typical",
    session_start_hour: int = 18,
    tz: str = "America/New_York",
    timestamp_col: str = "timestamp",
) -> pd.DataFrame:
    """
    Add a futures-style continuous session VWAP.

    The VWAP resets at `session_start_hour` in local timezone, so it includes:
    - overnight
    - premarket
    - regular US session

    Example for CME equity index futures:
    session_start_hour = 18 means the session runs approximately
    from 18:00 ET to 17:59:59 ET next day.
    """
    out = df.copy()

    ts = out[timestamp_col]
    if ts.dt.tz is None:
        ts_local = ts.dt.tz_localize(tz)
    else:
        ts_local = ts.dt.tz_convert(tz)
    out["_ts_local"] = ts_local

    # Build continuous session date by shifting all bars from session_start_hour or later to next calendar day.
    session_date = out["_ts_local"].dt.date
    mask_next_session = out["_ts_local"].dt.hour >= session_start_hour
    session_date = session_date.where(~mask_next_session, (out["_ts_local"] + pd.Timedelta(days=1)).dt.date)
    out["continuous_session_date"] = session_date

    if price_mode == "close":
        price = out["close"]
    elif price_mode == "typical":
        price = (out["high"] + out["low"] + out["close"]) / 3.0
    else:
        raise ValueError("price_mode must be 'close' or 'typical'.")

    volume = out["volume"].fillna(0.0)
    pv = price * volume

    cumulative_pv = pv.groupby(out["continuous_session_date"]).cumsum()
    cumulative_volume = volume.groupby(out["continuous_session_date"]).cumsum()

    out["continuous_session_vwap"] = np.where(
        cumulative_volume > 0,
        cumulative_pv / cumulative_volume,
        np.nan,
    )

    out = out.drop(columns=["_ts_local"])
    return out

def compute_opening_range(
    df: pd.DataFrame,
    or_minutes: int = 30,
    opening_time: str = "09:00:00",
) -> pd.DataFrame:
    """Compute opening range high/low/width/midpoint from the configured opening time."""
    out = df.copy()
    out["session_date"] = out["timestamp"].dt.date

    range_frames = []
    for session_date, group in out.groupby("session_date", sort=True):
        if group.empty:
            continue

        start = build_session_time(group["timestamp"].iloc[0], opening_time)
        end = start + pd.Timedelta(minutes=or_minutes)
        opening_window = group[(group["timestamp"] >= start) & (group["timestamp"] < end)]
        if opening_window.empty:
            continue

        high = opening_window["high"].max()
        low = opening_window["low"].min()
        width = high - low
        midpoint = (high + low) / 2.0
        range_frames.append(
            {
                "session_date": session_date,
                "or_high": high,
                "or_low": low,
                "or_width": width,
                "or_midpoint": midpoint,
            }
        )

    or_df = pd.DataFrame(range_frames)
    return out.merge(or_df, on="session_date", how="left")

def _normalize_windows(window: int | Iterable[int]) -> list[int]:
    if isinstance(window, (int, np.integer)):
        windows = [int(window)]
    else:
        windows = [int(value) for value in window]

    clean = sorted({value for value in windows if value > 0})
    if not clean:
        raise ValueError("ATR window must contain at least one positive integer.")
    return clean

def add_atr(df: pd.DataFrame, window: int | Iterable[int] = 14) -> pd.DataFrame:
    """Add simplified ATR feature(s) for one or many rolling windows."""
    out = df.copy()
    prev_close = out["close"].shift(1)
    tr = np.maximum(
        out["high"] - out["low"],
        np.maximum((out["high"] - prev_close).abs(), (out["low"] - prev_close).abs()),
    )
    for w in _normalize_windows(window):
        out[f"atr_{w}"] = tr.rolling(w).mean()
    return out

TRADE_LOG_COLUMNS = [
    "trade_id",
    "session_date",
    "direction",
    "quantity",
    "entry_time",
    "entry_price",
    "stop_price",
    "target_price",
    "exit_time",
    "exit_price",
    "exit_reason",
    "account_size_usd",
    "risk_per_trade_pct",
    "risk_budget_usd",
    "risk_per_contract_usd",
    "actual_risk_usd",
    "trade_risk_usd",
    "notional_usd",
    "leverage_used",
    "pnl_points",
    "pnl_ticks",
    "pnl_usd",
    "fees",
    "net_pnl_usd",
]

def empty_trade_log() -> pd.DataFrame:
    """Return an empty trade log dataframe with standard columns."""
    return pd.DataFrame(columns=TRADE_LOG_COLUMNS)

def trade_to_record(trade_id: int, data: dict[str, Any]) -> dict[str, Any]:
    """Build standardized trade record dict."""
    row = {col: None for col in TRADE_LOG_COLUMNS}
    row["trade_id"] = trade_id
    row.update(data)
    return row

@dataclass
class ExecutionModel:
    """Simple fixed commission + fixed slippage model."""

    commission_per_side_usd: float = DEFAULT_COMMISSION_PER_SIDE_USD
    slippage_ticks: float = float(DEFAULT_SLIPPAGE_TICKS)
    tick_size: float = DEFAULT_TICK_SIZE

    def apply_slippage(self, price: float, direction: int, is_entry: bool) -> float:
        """Adjust price by slippage in unfavorable direction."""
        slippage_points = self.slippage_ticks * self.tick_size
        if direction == 1:
            return price + slippage_points if is_entry else price - slippage_points
        return price - slippage_points if is_entry else price + slippage_points

    def round_trip_fees(self, quantity: int = 1) -> float:
        """Return total round-trip fees in USD for the requested quantity."""
        return 2.0 * self.commission_per_side_usd * quantity

EQUITY_TICK_SIZE = 0.01

EQUITY_POINT_VALUE_USD = 1.0

EQUITY_PAPER_COMMISSION_PER_SHARE = 0.0005

@dataclass(frozen=True)
class InstrumentDetails:
    """Execution metadata inferred from the selected dataset symbol."""

    symbol: str
    asset_class: str
    tick_size: float
    tick_value_usd: float
    point_value_usd: float
    commission_per_side_usd: float
    slippage_ticks: int

def resolve_instrument_details(symbol: str) -> InstrumentDetails:
    """Resolve instrument specifications from repo defaults or equity-like fallback."""
    upper_symbol = symbol.upper()
    if upper_symbol in INSTRUMENT_SPECS:
        spec = INSTRUMENT_SPECS[upper_symbol]
        return InstrumentDetails(
            symbol=upper_symbol,
            asset_class="futures",
            tick_size=float(spec["tick_size"]),
            tick_value_usd=float(spec["tick_value_usd"]),
            point_value_usd=float(spec["point_value_usd"]),
            commission_per_side_usd=float(spec["commission_per_side_usd"]),
            slippage_ticks=int(spec["slippage_ticks"]),
        )

    return InstrumentDetails(
        symbol=upper_symbol,
        asset_class="equity",
        tick_size=EQUITY_TICK_SIZE,
        tick_value_usd=EQUITY_TICK_SIZE * EQUITY_POINT_VALUE_USD,
        point_value_usd=EQUITY_POINT_VALUE_USD,
        commission_per_side_usd=EQUITY_PAPER_COMMISSION_PER_SHARE,
        slippage_ticks=0,
    )

def build_execution_model_for_profile(
    symbol: str,
    profile_name: str,
    cost_multiplier: float = 1.0,
) -> tuple[ExecutionModel, InstrumentDetails]:
    """Build the execution model used by a VWAP run."""
    details = resolve_instrument_details(symbol)
    if cost_multiplier <= 0:
        raise ValueError("cost_multiplier must be strictly positive.")

    if profile_name == "repo_realistic":
        commission = details.commission_per_side_usd * cost_multiplier
        slippage_ticks = int(round(details.slippage_ticks * cost_multiplier))
        return (
            ExecutionModel(
                commission_per_side_usd=commission,
                slippage_ticks=slippage_ticks,
                tick_size=details.tick_size,
            ),
            details,
        )

    if profile_name == "paper_reference":
        if details.asset_class == "futures":
            commission = details.commission_per_side_usd * cost_multiplier
            slippage_ticks = int(round(details.slippage_ticks * cost_multiplier))
            return (
                ExecutionModel(
                    commission_per_side_usd=commission,
                    slippage_ticks=slippage_ticks,
                    tick_size=details.tick_size,
                ),
                details,
            )
        return (
            ExecutionModel(
                commission_per_side_usd=EQUITY_PAPER_COMMISSION_PER_SHARE * cost_multiplier,
                slippage_ticks=0,
                tick_size=details.tick_size,
            ),
            details,
        )

    raise ValueError(f"Unsupported execution profile '{profile_name}'.")

def latest_path_for_symbol(symbol: str, input_paths: dict[str, Path] | None = None) -> Path:
    if input_paths is not None and symbol in input_paths:
        return Path(input_paths[symbol])
    files = sorted((REPO_ROOT / "data" / "processed" / "parquet").glob(f"{symbol}_c_0_1m_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No input dataset found for {symbol} in data/processed/parquet.")
    return files[-1]

def load_symbol_data(symbol: str, input_paths: dict[str, Path] | None = None) -> pd.DataFrame:
    source_path = latest_path_for_symbol(symbol, input_paths=input_paths)
    return clean_ohlcv(load_ohlcv_file(source_path))

def resample_rth_1h(raw: pd.DataFrame) -> pd.DataFrame:
    scoped = extract_rth(raw.copy())
    if scoped.empty:
        return scoped
    scoped["timestamp"] = pd.to_datetime(scoped["timestamp"], errors="coerce")
    scoped = scoped.set_index("timestamp").sort_index()
    bars = scoped.resample("1h", label="left", closed="left", origin="start_day", offset="30min").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    return bars.dropna(subset=["open", "high", "low", "close"]).reset_index()

def split_sessions(frame: pd.DataFrame, ratio: float = 0.7) -> tuple[list, list]:
    sessions = sorted(pd.to_datetime(frame["session_date"]).dt.date.unique())
    if len(sessions) < 2:
        raise ValueError("Need at least two sessions to build an IS/OOS split.")
    cut = max(1, int(len(sessions) * ratio))
    cut = min(cut, len(sessions) - 1)
    return sessions[:cut], sessions[cut:]


In [ ]:
@dataclass(frozen=True)
class VolumeClimaxPullbackV2Variant:
    """Compact research config for the V2 standalone campaign."""

    name: str
    family: str
    timeframe: str
    volume_quantile: float
    volume_lookback: int
    min_body_fraction: float
    min_range_atr: float
    trend_ema_window: int | None
    ema_slope_threshold: float | None
    atr_percentile_low: float | None
    atr_percentile_high: float | None
    compression_ratio_max: float | None
    entry_mode: str
    pullback_fraction: float | None
    confirmation_window: int | None
    exit_mode: str
    rr_target: float
    atr_target_multiple: float | None
    time_stop_bars: int
    trailing_atr_multiple: float
    session_overlay: str = "all_rth"

def _rolling_percent_rank(series: pd.Series, window: int) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    out = np.full(len(values), np.nan, dtype=float)
    if window <= 1:
        return pd.Series(out, index=series.index, dtype=float)

    for idx in range(window - 1, len(values)):
        window_values = values[idx - window + 1 : idx + 1]
        if np.isnan(window_values).any():
            continue
        out[idx] = float(np.searchsorted(np.sort(window_values), window_values[-1], side="right") / window)
    return pd.Series(out, index=series.index, dtype=float)

def prepare_volume_climax_pullback_v2_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build strict historical features for the 1h standalone campaign."""

    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    out = out.sort_values("timestamp").reset_index(drop=True)
    out = add_session_date(out)

    open_ = pd.to_numeric(out["open"], errors="coerce")
    high = pd.to_numeric(out["high"], errors="coerce")
    low = pd.to_numeric(out["low"], errors="coerce")
    close = pd.to_numeric(out["close"], errors="coerce")
    volume = pd.to_numeric(out["volume"], errors="coerce").fillna(0.0)

    prev_close = close.shift(1)
    true_range = pd.concat(
        [(high - low), (high - prev_close).abs(), (low - prev_close).abs()],
        axis=1,
    ).max(axis=1)

    out["atr_5"] = true_range.rolling(5, min_periods=5).mean()
    out["atr_20"] = true_range.rolling(20, min_periods=20).mean()
    out["atr_50"] = true_range.rolling(50, min_periods=50).mean()
    out["bar_range"] = high - low
    out["body"] = (close - open_).abs()
    out["body_fraction"] = out["body"] / out["bar_range"].replace(0.0, np.nan)
    out["range_atr"] = out["bar_range"] / out["atr_20"].replace(0.0, np.nan)
    out["atr_ratio_5_20"] = out["atr_5"] / out["atr_20"].replace(0.0, np.nan)

    out["ema20"] = close.ewm(span=20, adjust=False).mean()
    out["ema50"] = close.ewm(span=50, adjust=False).mean()
    out["ema20_slope_3_atr"] = (out["ema20"] - out["ema20"].shift(3)) / out["atr_20"].replace(0.0, np.nan)
    out["ema50_slope_3_atr"] = (out["ema50"] - out["ema50"].shift(3)) / out["atr_20"].replace(0.0, np.nan)

    typical = (high + low + close) / 3.0
    session_key = pd.to_datetime(out["session_date"])
    cum_volume = volume.groupby(session_key, sort=True).cumsum().replace(0.0, np.nan)
    out["session_vwap"] = (typical * volume).groupby(session_key, sort=True).cumsum() / cum_volume
    out["atr_percentile_100"] = _rolling_percent_rank(out["atr_20"], 100)
    out["is_last_bar_of_session"] = out["session_date"] != out["session_date"].shift(-1)
    return out

def build_volume_climax_pullback_v2_signal_frame(
    features: pd.DataFrame,
    variant: VolumeClimaxPullbackV2Variant,
) -> pd.DataFrame:
    """Project a raw climax setup into a strict t-1 signal frame."""

    out = features.copy()
    volume = pd.to_numeric(out["volume"], errors="coerce").fillna(0.0)
    volume_threshold = volume.shift(1).rolling(
        int(variant.volume_lookback),
        min_periods=int(variant.volume_lookback),
    ).quantile(float(variant.volume_quantile))

    prev_open = pd.to_numeric(out["open"], errors="coerce").shift(1)
    prev_high = pd.to_numeric(out["high"], errors="coerce").shift(1)
    prev_low = pd.to_numeric(out["low"], errors="coerce").shift(1)
    prev_close = pd.to_numeric(out["close"], errors="coerce").shift(1)
    prev_body_fraction = pd.to_numeric(out["body_fraction"], errors="coerce").shift(1)
    prev_range_atr = pd.to_numeric(out["range_atr"], errors="coerce").shift(1)

    bullish_prev = prev_close > prev_open
    bearish_prev = prev_close < prev_open
    climax_prev = volume.shift(1) > volume_threshold
    quality_prev = (prev_body_fraction >= float(variant.min_body_fraction)) & (
        prev_range_atr >= float(variant.min_range_atr)
    )

    raw_short = climax_prev & bullish_prev & quality_prev
    raw_long = climax_prev & bearish_prev & quality_prev

    short_regime = pd.Series(True, index=out.index, dtype=bool)
    long_regime = pd.Series(True, index=out.index, dtype=bool)

    if variant.trend_ema_window is not None and variant.ema_slope_threshold is not None:
        slope_col = f"ema{int(variant.trend_ema_window)}_slope_3_atr"
        slope_prev = pd.to_numeric(out[slope_col], errors="coerce").shift(1)
        threshold = float(variant.ema_slope_threshold)
        short_regime &= slope_prev <= threshold
        long_regime &= slope_prev >= -threshold

    if variant.atr_percentile_low is not None and variant.atr_percentile_high is not None:
        atr_pct_prev = pd.to_numeric(out["atr_percentile_100"], errors="coerce").shift(1)
        short_regime &= atr_pct_prev.between(float(variant.atr_percentile_low), float(variant.atr_percentile_high))
        long_regime &= atr_pct_prev.between(float(variant.atr_percentile_low), float(variant.atr_percentile_high))

    if variant.compression_ratio_max is not None:
        compression_prev = pd.to_numeric(out["atr_ratio_5_20"], errors="coerce").shift(1)
        short_regime &= compression_prev <= float(variant.compression_ratio_max)
        long_regime &= compression_prev <= float(variant.compression_ratio_max)

    signal = pd.Series(0, index=out.index, dtype=int)
    signal.loc[raw_short & short_regime] = -1
    signal.loc[raw_long & long_regime] = 1

    raw_signal = pd.Series(0, index=out.index, dtype=int)
    raw_signal.loc[raw_short] = -1
    raw_signal.loc[raw_long] = 1

    out["volume_threshold_hist"] = volume_threshold
    out["raw_signal"] = raw_signal
    out["signal"] = signal
    out["entry_short"] = signal.eq(-1)
    out["entry_long"] = signal.eq(1)
    out["setup_signal_time"] = out["timestamp"].shift(1)
    out["setup_reference_open"] = prev_open
    out["setup_reference_high"] = prev_high
    out["setup_reference_low"] = prev_low
    out["setup_reference_close"] = prev_close
    out["setup_reference_range"] = (prev_high - prev_low).clip(lower=0.0)
    out["setup_reference_atr"] = pd.to_numeric(out["atr_20"], errors="coerce").shift(1)
    out["setup_reference_vwap"] = pd.to_numeric(out["session_vwap"], errors="coerce").shift(1)
    out["setup_stop_reference_long"] = prev_low
    out["setup_stop_reference_short"] = prev_high
    return out

@dataclass(frozen=True)
class FixedContractPositionSizing:
    """Always trade a fixed number of contracts."""

    fixed_contracts: int = 1

@dataclass(frozen=True)
class RiskPercentPositionSizing:
    """Size each trade from a fixed fraction of current equity."""

    initial_capital_usd: float
    risk_pct: float
    max_contracts: int
    skip_trade_if_too_small: bool = True
    compound_realized_pnl: bool = True

PositionSizingConfig = FixedContractPositionSizing | RiskPercentPositionSizing

@dataclass(frozen=True)
class PositionSizingDecision:
    """Resolved contract count and audit fields for a trade attempt."""

    position_sizing_mode: str
    contracts: int
    contracts_raw: float | None
    capital_before_trade_usd: float | None
    risk_pct: float | None
    risk_budget_usd: float | None
    stop_distance_points: float | None
    risk_per_contract_usd: float | None
    actual_risk_usd: float | None
    notional_usd: float | None
    leverage_used: float | None
    max_contracts: int | None
    skip_trade_if_too_small: bool | None
    skipped: bool
    skip_reason: str | None

def validate_position_sizing(config: PositionSizingConfig | None) -> None:
    """Validate a sizing configuration."""
    if config is None:
        return

    if isinstance(config, FixedContractPositionSizing):
        if int(config.fixed_contracts) < 1:
            raise ValueError("fixed_contracts must be at least 1.")
        return

    if float(config.initial_capital_usd) <= 0:
        raise ValueError("initial_capital_usd must be strictly positive.")
    if float(config.risk_pct) <= 0 or float(config.risk_pct) > 1.0:
        raise ValueError("risk_pct must be in the (0, 1] interval.")
    if int(config.max_contracts) < 1:
        raise ValueError("max_contracts must be at least 1.")

def initial_capital_from_sizing(config: PositionSizingConfig | None) -> float | None:
    """Return the initial capital implied by the sizing config, when any."""
    if isinstance(config, RiskPercentPositionSizing):
        return float(config.initial_capital_usd)
    return None

def compounds_realized_pnl(config: PositionSizingConfig | None) -> bool:
    """Return whether risk-percent sizing should compound realized PnL."""
    return isinstance(config, RiskPercentPositionSizing) and bool(config.compound_realized_pnl)

def resolve_position_size(
    *,
    config: PositionSizingConfig | None,
    capital_before_trade_usd: float | None,
    entry_price: float,
    initial_stop_price: float,
    point_value_usd: float,
) -> PositionSizingDecision:
    """Return the contract count and the audit trail for a trade attempt."""
    validate_position_sizing(config)

    stop_distance_points = abs(float(entry_price) - float(initial_stop_price))
    risk_per_contract_usd = (
        stop_distance_points * float(point_value_usd)
        if math.isfinite(stop_distance_points) and math.isfinite(float(point_value_usd))
        else float("nan")
    )

    if config is None:
        config = FixedContractPositionSizing()

    if isinstance(config, FixedContractPositionSizing):
        contracts = int(config.fixed_contracts)
        notional_usd = float(contracts) * float(entry_price) * float(point_value_usd)
        leverage_used = (
            notional_usd / float(capital_before_trade_usd)
            if capital_before_trade_usd is not None and float(capital_before_trade_usd) > 0
            else None
        )
        actual_risk_usd = (
            float(contracts) * float(risk_per_contract_usd)
            if math.isfinite(risk_per_contract_usd) and float(risk_per_contract_usd) > 0
            else None
        )
        return PositionSizingDecision(
            position_sizing_mode="fixed_contracts",
            contracts=contracts,
            contracts_raw=float(contracts),
            capital_before_trade_usd=capital_before_trade_usd,
            risk_pct=None,
            risk_budget_usd=None,
            stop_distance_points=stop_distance_points if math.isfinite(stop_distance_points) else None,
            risk_per_contract_usd=float(risk_per_contract_usd) if math.isfinite(risk_per_contract_usd) else None,
            actual_risk_usd=actual_risk_usd,
            notional_usd=notional_usd,
            leverage_used=leverage_used,
            max_contracts=None,
            skip_trade_if_too_small=None,
            skipped=False,
            skip_reason=None,
        )

    if capital_before_trade_usd is None or float(capital_before_trade_usd) <= 0:
        return PositionSizingDecision(
            position_sizing_mode="risk_percent",
            contracts=0,
            contracts_raw=None,
            capital_before_trade_usd=capital_before_trade_usd,
            risk_pct=float(config.risk_pct),
            risk_budget_usd=None,
            stop_distance_points=stop_distance_points if math.isfinite(stop_distance_points) else None,
            risk_per_contract_usd=float(risk_per_contract_usd) if math.isfinite(risk_per_contract_usd) else None,
            actual_risk_usd=None,
            notional_usd=None,
            leverage_used=None,
            max_contracts=int(config.max_contracts),
            skip_trade_if_too_small=bool(config.skip_trade_if_too_small),
            skipped=True,
            skip_reason="non_positive_capital",
        )

    risk_budget_usd = float(capital_before_trade_usd) * float(config.risk_pct)
    if not math.isfinite(risk_per_contract_usd) or float(risk_per_contract_usd) <= 0:
        return PositionSizingDecision(
            position_sizing_mode="risk_percent",
            contracts=0,
            contracts_raw=None,
            capital_before_trade_usd=float(capital_before_trade_usd),
            risk_pct=float(config.risk_pct),
            risk_budget_usd=risk_budget_usd,
            stop_distance_points=stop_distance_points if math.isfinite(stop_distance_points) else None,
            risk_per_contract_usd=float(risk_per_contract_usd) if math.isfinite(risk_per_contract_usd) else None,
            actual_risk_usd=None,
            notional_usd=None,
            leverage_used=None,
            max_contracts=int(config.max_contracts),
            skip_trade_if_too_small=bool(config.skip_trade_if_too_small),
            skipped=True,
            skip_reason="non_positive_risk_per_contract",
        )

    contracts_raw = float(risk_budget_usd) / float(risk_per_contract_usd)
    contracts = int(math.floor(contracts_raw))
    skip_reason: str | None = None

    if contracts < 1:
        if bool(config.skip_trade_if_too_small):
            skip_reason = "contracts_below_one"
            contracts = 0
        else:
            contracts = 1

    contracts = min(int(contracts), int(config.max_contracts))
    if contracts < 1:
        skip_reason = skip_reason or "contracts_below_one_after_cap"

    notional_usd = float(contracts) * float(entry_price) * float(point_value_usd) if contracts > 0 else None
    leverage_used = notional_usd / float(capital_before_trade_usd) if contracts > 0 else None
    actual_risk_usd = float(contracts) * float(risk_per_contract_usd) if contracts > 0 else None
    return PositionSizingDecision(
        position_sizing_mode="risk_percent",
        contracts=int(contracts),
        contracts_raw=contracts_raw,
        capital_before_trade_usd=float(capital_before_trade_usd),
        risk_pct=float(config.risk_pct),
        risk_budget_usd=risk_budget_usd,
        stop_distance_points=float(stop_distance_points),
        risk_per_contract_usd=float(risk_per_contract_usd),
        actual_risk_usd=actual_risk_usd,
        notional_usd=notional_usd,
        leverage_used=leverage_used,
        max_contracts=int(config.max_contracts),
        skip_trade_if_too_small=bool(config.skip_trade_if_too_small),
        skipped=bool(contracts < 1),
        skip_reason=skip_reason,
    )

@dataclass(frozen=True)
class VolumeClimaxPullbackV2BacktestResult:
    trades: pd.DataFrame
    sizing_decisions: pd.DataFrame = field(default_factory=pd.DataFrame)

SIZING_DECISION_COLUMNS = [
    "variant_name",
    "session_date",
    "entry_time",
    "entry_signal_time",
    "direction",
    "entry_price",
    "initial_stop_price",
    "position_sizing_mode",
    "capital_before_trade_usd",
    "risk_pct",
    "risk_budget_usd",
    "stop_distance_points",
    "risk_per_contract_usd",
    "contracts_raw",
    "contracts",
    "actual_risk_usd",
    "notional_usd",
    "leverage_used",
    "max_contracts",
    "skip_trade_if_too_small",
    "skipped",
    "skip_reason",
]

def _float_or_nan(value: Any) -> float:
    if value is None:
        return float("nan")
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value) if not pd.isna(value) else float("nan")
    try:
        parsed = pd.to_numeric(value, errors="coerce")
    except (TypeError, ValueError):
        return float("nan")
    return float(parsed) if pd.notna(parsed) else float("nan")

def _bool_or_false(value: Any) -> bool:
    return bool(value) if value is not None and not pd.isna(value) else False

def _empty_sizing_decision_log() -> pd.DataFrame:
    return pd.DataFrame(columns=SIZING_DECISION_COLUMNS)

def _entry_market_fill(execution_model: ExecutionModel, raw_price: float, direction: int) -> float:
    return float(execution_model.apply_slippage(raw_price, direction, is_entry=True))

def _entry_limit_fill(raw_open: float, limit_price: float, direction: int) -> float:
    if direction == 1:
        return float(raw_open) if raw_open <= limit_price else float(limit_price)
    return float(raw_open) if raw_open >= limit_price else float(limit_price)

def _resolve_target_price(
    variant: VolumeClimaxPullbackV2Variant,
    direction: int,
    entry_price: float,
    risk_points: float,
    reference_atr: float,
    reference_vwap: float,
) -> tuple[float | None, float | None, float | None]:
    if variant.exit_mode == "fixed_rr":
        target = entry_price + direction * float(variant.rr_target) * risk_points
        return float(target), None, None

    if variant.exit_mode == "target_vwap":
        if not np.isfinite(reference_vwap):
            return None, None, None
        favorable = (reference_vwap - entry_price) * direction
        if favorable <= 0:
            return None, None, None
        return float(reference_vwap), None, None

    if variant.exit_mode == "atr_fraction":
        if not np.isfinite(reference_atr) or reference_atr <= 0 or variant.atr_target_multiple is None:
            return None, None, None
        target = entry_price + direction * float(variant.atr_target_multiple) * reference_atr
        return float(target), None, None

    if variant.exit_mode == "mixed":
        if not np.isfinite(reference_atr) or reference_atr <= 0:
            return None, None, None
        partial_target = entry_price + direction * 0.5 * reference_atr
        trailing_offset = float(variant.trailing_atr_multiple) * reference_atr
        return None, float(partial_target), float(trailing_offset)

    raise ValueError(f"Unsupported exit_mode '{variant.exit_mode}'.")

def _append_sizing_decision(
    sizing_decisions: list[dict[str, Any]],
    *,
    variant: VolumeClimaxPullbackV2Variant,
    session_date: Any,
    entry_time: pd.Timestamp,
    entry_signal_time: pd.Timestamp | None,
    direction: int,
    entry_price: float,
    initial_stop_price: float | None,
    decision: PositionSizingDecision | None = None,
    skip_reason: str | None = None,
    position_sizing_mode: str | None = None,
) -> None:
    record = {column: np.nan for column in SIZING_DECISION_COLUMNS}
    record.update(
        {
            "variant_name": variant.name,
            "session_date": session_date,
            "entry_time": entry_time,
            "entry_signal_time": entry_signal_time,
            "direction": "long" if int(direction) == 1 else "short",
            "entry_price": float(entry_price),
            "initial_stop_price": float(initial_stop_price) if initial_stop_price is not None and np.isfinite(initial_stop_price) else np.nan,
            "position_sizing_mode": position_sizing_mode,
            "skipped": True,
            "skip_reason": skip_reason,
        }
    )
    if initial_stop_price is not None and np.isfinite(initial_stop_price):
        record["stop_distance_points"] = abs(float(entry_price) - float(initial_stop_price))
    if decision is not None:
        record.update(
            {
                "position_sizing_mode": decision.position_sizing_mode,
                "capital_before_trade_usd": decision.capital_before_trade_usd,
                "risk_pct": decision.risk_pct,
                "risk_budget_usd": decision.risk_budget_usd,
                "stop_distance_points": decision.stop_distance_points,
                "risk_per_contract_usd": decision.risk_per_contract_usd,
                "contracts_raw": decision.contracts_raw,
                "contracts": int(decision.contracts),
                "actual_risk_usd": decision.actual_risk_usd,
                "notional_usd": decision.notional_usd,
                "leverage_used": decision.leverage_used,
                "max_contracts": decision.max_contracts,
                "skip_trade_if_too_small": decision.skip_trade_if_too_small,
                "skipped": bool(decision.skipped),
                "skip_reason": decision.skip_reason,
            }
        )
    sizing_decisions.append(record)

def _trade_record(
    trade_id: int,
    exit_time: pd.Timestamp,
    exit_price: float,
    exit_reason: str,
    instrument: InstrumentDetails,
    execution_model: ExecutionModel,
    variant: VolumeClimaxPullbackV2Variant,
    open_trade: dict[str, Any],
) -> dict[str, Any]:
    direction = int(open_trade["direction"])
    quantity = int(open_trade["quantity"])
    entry_price = float(open_trade["entry_price"])
    pnl_points = (float(exit_price) - float(entry_price)) * direction
    gross = pnl_points * float(instrument.point_value_usd) * float(quantity)
    fees = execution_model.round_trip_fees(quantity=quantity)
    net = gross - fees
    capital_before_trade = _float_or_nan(open_trade.get("account_size_usd"))
    capital_after_trade = capital_before_trade + float(net) if np.isfinite(capital_before_trade) else np.nan
    record = trade_to_record(
        trade_id,
        {
            "session_date": open_trade["session_date"],
            "direction": "long" if direction == 1 else "short",
            "quantity": int(quantity),
            "entry_time": open_trade["entry_time"],
            "entry_price": float(entry_price),
            "stop_price": float(open_trade["stop_price"]),
            "target_price": float(open_trade["target_price"])
            if np.isfinite(_float_or_nan(open_trade.get("target_price")))
            else np.nan,
            "exit_time": exit_time,
            "exit_price": float(exit_price),
            "exit_reason": exit_reason,
            "account_size_usd": capital_before_trade,
            "risk_per_trade_pct": _float_or_nan(open_trade.get("risk_per_trade_pct")),
            "risk_budget_usd": _float_or_nan(open_trade.get("risk_budget_usd")),
            "risk_per_contract_usd": _float_or_nan(open_trade.get("risk_per_contract_usd")),
            "actual_risk_usd": _float_or_nan(open_trade.get("actual_risk_usd")),
            "trade_risk_usd": _float_or_nan(open_trade.get("trade_risk_usd")),
            "notional_usd": _float_or_nan(open_trade.get("notional_usd")),
            "leverage_used": _float_or_nan(open_trade.get("leverage_used")),
            "pnl_points": float(pnl_points),
            "pnl_ticks": float(pnl_points / instrument.tick_size) if instrument.tick_size > 0 else np.nan,
            "pnl_usd": float(gross),
            "fees": float(fees),
            "net_pnl_usd": float(net),
        },
    )
    record["variant_name"] = variant.name
    record["entry_signal_time"] = open_trade.get("entry_signal_time")
    record["bars_held"] = int(open_trade["bars_held"])
    record["position_sizing_mode"] = open_trade.get("position_sizing_mode")
    record["risk_pct_decimal"] = _float_or_nan(open_trade.get("risk_pct"))
    record["stop_distance_points"] = _float_or_nan(open_trade.get("stop_distance_points"))
    record["contracts_raw"] = _float_or_nan(open_trade.get("contracts_raw"))
    record["max_contracts"] = _float_or_nan(open_trade.get("max_contracts"))
    record["skip_trade_if_too_small"] = open_trade.get("skip_trade_if_too_small")
    record["capital_after_trade_usd"] = capital_after_trade
    record["initial_stop_price"] = _float_or_nan(open_trade.get("initial_stop_price"))
    return record

def _build_open_trade(
    *,
    row: pd.Series,
    direction: int,
    entry_price: float,
    variant: VolumeClimaxPullbackV2Variant,
    instrument: InstrumentDetails,
    position_sizing: PositionSizingConfig | None,
    capital_before_trade_usd: float | None,
    sizing_decisions: list[dict[str, Any]],
) -> dict[str, Any] | None:
    stop_col = "setup_stop_reference_long" if direction == 1 else "setup_stop_reference_short"
    stop_price = _float_or_nan(row.get(stop_col))
    reference_atr = _float_or_nan(row.get("setup_reference_atr"))
    reference_vwap = _float_or_nan(row.get("setup_reference_vwap"))
    entry_time = pd.Timestamp(row["timestamp"])
    signal_time_raw = row.get("setup_signal_time")
    entry_signal_time = pd.Timestamp(signal_time_raw) if pd.notna(signal_time_raw) else None
    session_date = row.get("session_date")

    if not np.isfinite(stop_price):
        _append_sizing_decision(
            sizing_decisions,
            variant=variant,
            session_date=session_date,
            entry_time=entry_time,
            entry_signal_time=entry_signal_time,
            direction=direction,
            entry_price=float(entry_price),
            initial_stop_price=None,
            skip_reason="invalid_initial_stop",
            position_sizing_mode="unresolved",
        )
        return None

    risk_points = (float(entry_price) - float(stop_price)) * int(direction)
    if risk_points <= 0:
        _append_sizing_decision(
            sizing_decisions,
            variant=variant,
            session_date=session_date,
            entry_time=entry_time,
            entry_signal_time=entry_signal_time,
            direction=direction,
            entry_price=float(entry_price),
            initial_stop_price=float(stop_price),
            skip_reason="non_positive_stop_distance",
            position_sizing_mode="unresolved",
        )
        return None

    target_price, partial_target_price, trailing_offset = _resolve_target_price(
        variant=variant,
        direction=direction,
        entry_price=float(entry_price),
        risk_points=float(risk_points),
        reference_atr=float(reference_atr),
        reference_vwap=float(reference_vwap),
    )
    if variant.exit_mode != "mixed" and target_price is None:
        _append_sizing_decision(
            sizing_decisions,
            variant=variant,
            session_date=session_date,
            entry_time=entry_time,
            entry_signal_time=entry_signal_time,
            direction=direction,
            entry_price=float(entry_price),
            initial_stop_price=float(stop_price),
            skip_reason="invalid_target",
            position_sizing_mode="unresolved",
        )
        return None

    sizing = resolve_position_size(
        config=position_sizing,
        capital_before_trade_usd=capital_before_trade_usd,
        entry_price=float(entry_price),
        initial_stop_price=float(stop_price),
        point_value_usd=float(instrument.point_value_usd),
    )
    _append_sizing_decision(
        sizing_decisions,
        variant=variant,
        session_date=session_date,
        entry_time=entry_time,
        entry_signal_time=entry_signal_time,
        direction=direction,
        entry_price=float(entry_price),
        initial_stop_price=float(stop_price),
        decision=sizing,
    )
    if sizing.skipped or int(sizing.contracts) < 1:
        return None

    return {
        "entry_time": entry_time,
        "entry_signal_time": entry_signal_time,
        "entry_price": float(entry_price),
        "direction": int(direction),
        "quantity": int(sizing.contracts),
        "stop_price": float(stop_price),
        "initial_stop_price": float(stop_price),
        "target_price": float(target_price) if target_price is not None else np.nan,
        "partial_target_price": float(partial_target_price) if partial_target_price is not None else np.nan,
        "trailing_offset": float(trailing_offset) if trailing_offset is not None else np.nan,
        "time_stop_bars": int(variant.time_stop_bars),
        "bars_held": 0,
        "account_size_usd": float(sizing.capital_before_trade_usd) if sizing.capital_before_trade_usd is not None else np.nan,
        "risk_per_trade_pct": float(sizing.risk_pct * 100.0) if sizing.risk_pct is not None else np.nan,
        "risk_pct": float(sizing.risk_pct) if sizing.risk_pct is not None else np.nan,
        "risk_budget_usd": float(sizing.risk_budget_usd) if sizing.risk_budget_usd is not None else np.nan,
        "risk_per_contract_usd": float(sizing.risk_per_contract_usd) if sizing.risk_per_contract_usd is not None else np.nan,
        "actual_risk_usd": float(sizing.actual_risk_usd) if sizing.actual_risk_usd is not None else np.nan,
        "trade_risk_usd": float(sizing.actual_risk_usd) if sizing.actual_risk_usd is not None else np.nan,
        "notional_usd": float(sizing.notional_usd) if sizing.notional_usd is not None else np.nan,
        "leverage_used": float(sizing.leverage_used) if sizing.leverage_used is not None else np.nan,
        "position_sizing_mode": sizing.position_sizing_mode,
        "contracts_raw": float(sizing.contracts_raw) if sizing.contracts_raw is not None else np.nan,
        "stop_distance_points": float(sizing.stop_distance_points) if sizing.stop_distance_points is not None else np.nan,
        "max_contracts": float(sizing.max_contracts) if sizing.max_contracts is not None else np.nan,
        "skip_trade_if_too_small": sizing.skip_trade_if_too_small,
        "reference_atr": float(reference_atr) if np.isfinite(reference_atr) else np.nan,
        "partial_filled": False,
        "remaining_fraction": 1.0,
        "realized_weighted_pnl_points": 0.0,
        "last_close": _float_or_nan(row.get("close")),
        "session_date": session_date,
    }

def _confirmation_triggered(
    row: pd.Series,
    direction: int,
    reference_close: float,
    reference_range: float,
    pullback_fraction: float,
) -> bool:
    if not np.isfinite(reference_close) or not np.isfinite(reference_range) or reference_range <= 0:
        return False

    high = _float_or_nan(row.get("high"))
    low = _float_or_nan(row.get("low"))
    open_ = _float_or_nan(row.get("open"))
    close = _float_or_nan(row.get("close"))
    pullback_distance = float(pullback_fraction) * float(reference_range)

    if direction == 1:
        retrace_level = float(reference_close) - pullback_distance
        return (low <= retrace_level) and (close > open_)

    retrace_level = float(reference_close) + pullback_distance
    return (high >= retrace_level) and (close < open_)

def _maybe_fill_pullback_limit(
    row: pd.Series,
    direction: int,
    reference_close: float,
    reference_range: float,
    pullback_fraction: float,
) -> float | None:
    if not np.isfinite(reference_close) or not np.isfinite(reference_range) or reference_range <= 0:
        return None

    open_ = _float_or_nan(row.get("open"))
    high = _float_or_nan(row.get("high"))
    low = _float_or_nan(row.get("low"))
    if direction == 1:
        limit_price = float(reference_close) - float(pullback_fraction) * float(reference_range)
        if low <= limit_price:
            return _entry_limit_fill(open_, limit_price, direction)
        return None

    limit_price = float(reference_close) + float(pullback_fraction) * float(reference_range)
    if high >= limit_price:
        return _entry_limit_fill(open_, limit_price, direction)
    return None

def _trail_stop_from_last_close(direction: int, stop_price: float, trailing_offset: float, last_close: float) -> float:
    if not np.isfinite(trailing_offset) or trailing_offset <= 0 or not np.isfinite(last_close):
        return float(stop_price)
    if direction == 1:
        return max(float(stop_price), float(last_close) - float(trailing_offset))
    return min(float(stop_price), float(last_close) + float(trailing_offset))

def _entry_row_from_pending_setup(current_row: pd.Series, pending_setup: dict[str, Any]) -> pd.Series:
    entry_row = current_row.copy()
    for key in (
        "setup_stop_reference_long",
        "setup_stop_reference_short",
        "setup_reference_atr",
        "setup_reference_vwap",
        "setup_signal_time",
    ):
        if key in pending_setup:
            entry_row[key] = pending_setup[key]
    return entry_row

def _finalize_trade(
    trades: list[dict[str, Any]],
    trade_id: int,
    exit_time: pd.Timestamp,
    raw_exit_price: float,
    exit_reason: str,
    execution_model: ExecutionModel,
    instrument: InstrumentDetails,
    variant: VolumeClimaxPullbackV2Variant,
    open_trade: dict[str, Any],
) -> int:
    direction = int(open_trade["direction"])
    entry_price = float(open_trade["entry_price"])
    fill_exit = float(execution_model.apply_slippage(raw_exit_price, direction, is_entry=False))
    weighted_final_points = float(open_trade["remaining_fraction"]) * (fill_exit - float(entry_price)) * int(direction)
    total_weighted_points = float(open_trade["realized_weighted_pnl_points"]) + weighted_final_points
    effective_exit_price = float(entry_price) + (float(total_weighted_points) * int(direction))
    trades.append(
        _trade_record(
            trade_id=trade_id,
            exit_time=exit_time,
            exit_price=effective_exit_price,
            exit_reason=exit_reason,
            instrument=instrument,
            execution_model=execution_model,
            variant=variant,
            open_trade=open_trade,
        )
    )
    return trade_id + 1

def run_volume_climax_pullback_v2_backtest(
    signal_df: pd.DataFrame,
    variant: VolumeClimaxPullbackV2Variant,
    execution_model: ExecutionModel,
    instrument: InstrumentDetails,
    position_sizing: PositionSizingConfig | None = None,
) -> VolumeClimaxPullbackV2BacktestResult:
    validate_position_sizing(position_sizing)
    trades: list[dict[str, Any]] = []
    sizing_decisions: list[dict[str, Any]] = []
    open_trade: dict[str, Any] | None = None
    trade_id = 1
    capital_before_trade_usd = initial_capital_from_sizing(position_sizing)
    compound_realized_pnl = compounds_realized_pnl(position_sizing)

    for session_date, sdf in signal_df.groupby("session_date", sort=True):
        sdf = sdf.sort_values("timestamp").reset_index(drop=True)
        pending_setup: dict[str, Any] | None = None
        last_idx = len(sdf) - 1

        for idx, row in sdf.iterrows():
            timestamp = pd.Timestamp(row["timestamp"])
            open_ = _float_or_nan(row.get("open"))
            high = _float_or_nan(row.get("high"))
            low = _float_or_nan(row.get("low"))
            close = _float_or_nan(row.get("close"))

            trade_was_open_at_bar_start = open_trade is not None

            if open_trade is not None:
                direction = int(open_trade["direction"])
                if variant.exit_mode == "mixed" and _bool_or_false(open_trade.get("partial_filled")):
                    open_trade["stop_price"] = _trail_stop_from_last_close(
                        direction=direction,
                        stop_price=float(open_trade["stop_price"]),
                        trailing_offset=_float_or_nan(open_trade.get("trailing_offset")),
                        last_close=_float_or_nan(open_trade.get("last_close")),
                    )

                open_trade["bars_held"] = int(open_trade["bars_held"]) + 1

                if variant.exit_mode == "mixed":
                    stop_hit = (low <= float(open_trade["stop_price"])) if direction == 1 else (high >= float(open_trade["stop_price"]))

                    if not _bool_or_false(open_trade.get("partial_filled")):
                        partial_target_price = _float_or_nan(open_trade.get("partial_target_price"))
                        partial_hit = (high >= partial_target_price) if direction == 1 else (low <= partial_target_price)

                        if stop_hit and partial_hit:
                            trade_id = _finalize_trade(
                                trades=trades,
                                trade_id=trade_id,
                                exit_time=timestamp,
                                raw_exit_price=float(open_trade["stop_price"]),
                                exit_reason="stop_ambiguous_first",
                                execution_model=execution_model,
                                instrument=instrument,
                                variant=variant,
                                open_trade=open_trade,
                            )
                            if compound_realized_pnl and capital_before_trade_usd is not None:
                                capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                            open_trade = None
                        elif stop_hit:
                            trade_id = _finalize_trade(
                                trades=trades,
                                trade_id=trade_id,
                                exit_time=timestamp,
                                raw_exit_price=float(open_trade["stop_price"]),
                                exit_reason="stop",
                                execution_model=execution_model,
                                instrument=instrument,
                                variant=variant,
                                open_trade=open_trade,
                            )
                            if compound_realized_pnl and capital_before_trade_usd is not None:
                                capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                            open_trade = None
                        elif partial_hit:
                            open_trade["realized_weighted_pnl_points"] = float(
                                open_trade["realized_weighted_pnl_points"]
                            ) + 0.5 * (float(partial_target_price) - float(open_trade["entry_price"])) * direction
                            open_trade["remaining_fraction"] = 0.5
                            open_trade["partial_filled"] = True
                            if direction == 1:
                                open_trade["stop_price"] = max(float(open_trade["stop_price"]), float(open_trade["entry_price"]))
                            else:
                                open_trade["stop_price"] = min(float(open_trade["stop_price"]), float(open_trade["entry_price"]))

                            if int(open_trade["bars_held"]) >= int(open_trade["time_stop_bars"]):
                                trade_id = _finalize_trade(
                                    trades=trades,
                                    trade_id=trade_id,
                                    exit_time=timestamp,
                                    raw_exit_price=close,
                                    exit_reason="mixed_partial_time_stop",
                                    execution_model=execution_model,
                                    instrument=instrument,
                                    variant=variant,
                                    open_trade=open_trade,
                                )
                                if compound_realized_pnl and capital_before_trade_usd is not None:
                                    capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                                open_trade = None
                            elif idx == last_idx:
                                trade_id = _finalize_trade(
                                    trades=trades,
                                    trade_id=trade_id,
                                    exit_time=timestamp,
                                    raw_exit_price=close,
                                    exit_reason="mixed_partial_eod_flat",
                                    execution_model=execution_model,
                                    instrument=instrument,
                                    variant=variant,
                                    open_trade=open_trade,
                                )
                                if compound_realized_pnl and capital_before_trade_usd is not None:
                                    capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                                open_trade = None
                    else:
                        if stop_hit:
                            trade_id = _finalize_trade(
                                trades=trades,
                                trade_id=trade_id,
                                exit_time=timestamp,
                                raw_exit_price=float(open_trade["stop_price"]),
                                exit_reason="mixed_trailing_stop",
                                execution_model=execution_model,
                                instrument=instrument,
                                variant=variant,
                                open_trade=open_trade,
                            )
                            if compound_realized_pnl and capital_before_trade_usd is not None:
                                capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                            open_trade = None
                        elif int(open_trade["bars_held"]) >= int(open_trade["time_stop_bars"]):
                            trade_id = _finalize_trade(
                                trades=trades,
                                trade_id=trade_id,
                                exit_time=timestamp,
                                raw_exit_price=close,
                                exit_reason="mixed_partial_time_stop",
                                execution_model=execution_model,
                                instrument=instrument,
                                variant=variant,
                                open_trade=open_trade,
                            )
                            if compound_realized_pnl and capital_before_trade_usd is not None:
                                capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                            open_trade = None
                        elif idx == last_idx:
                            trade_id = _finalize_trade(
                                trades=trades,
                                trade_id=trade_id,
                                exit_time=timestamp,
                                raw_exit_price=close,
                                exit_reason="mixed_partial_eod_flat",
                                execution_model=execution_model,
                                instrument=instrument,
                                variant=variant,
                                open_trade=open_trade,
                            )
                            if compound_realized_pnl and capital_before_trade_usd is not None:
                                capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                            open_trade = None
                else:
                    stop_hit = (low <= float(open_trade["stop_price"])) if direction == 1 else (high >= float(open_trade["stop_price"]))
                    target_hit = (high >= float(open_trade["target_price"])) if direction == 1 else (low <= float(open_trade["target_price"]))
                    exit_price = None
                    exit_reason = None

                    if stop_hit and target_hit:
                        exit_price = float(open_trade["stop_price"])
                        exit_reason = "stop_ambiguous_first"
                    elif stop_hit:
                        exit_price = float(open_trade["stop_price"])
                        exit_reason = "stop"
                    elif target_hit:
                        exit_price = float(open_trade["target_price"])
                        exit_reason = "target"
                    elif int(open_trade["bars_held"]) >= int(open_trade["time_stop_bars"]):
                        exit_price = close
                        exit_reason = "time_stop"
                    elif idx == last_idx:
                        exit_price = close
                        exit_reason = "eod_flat"

                    if exit_price is not None:
                        trade_id = _finalize_trade(
                            trades=trades,
                            trade_id=trade_id,
                            exit_time=timestamp,
                            raw_exit_price=float(exit_price),
                            exit_reason=str(exit_reason),
                            execution_model=execution_model,
                            instrument=instrument,
                            variant=variant,
                            open_trade=open_trade,
                        )
                        if compound_realized_pnl and capital_before_trade_usd is not None:
                            capital_before_trade_usd += float(trades[-1]["net_pnl_usd"])
                        open_trade = None

                if open_trade is not None:
                    open_trade["last_close"] = close

            if trade_was_open_at_bar_start:
                continue

            if open_trade is None and pending_setup is not None and _bool_or_false(pending_setup.get("confirmed_for_next_open")):
                open_trade = _build_open_trade(
                    row=_entry_row_from_pending_setup(row, pending_setup),
                    direction=int(pending_setup["direction"]),
                    entry_price=_entry_market_fill(execution_model, open_, int(pending_setup["direction"])),
                    variant=variant,
                    instrument=instrument,
                    position_sizing=position_sizing,
                    capital_before_trade_usd=capital_before_trade_usd,
                    sizing_decisions=sizing_decisions,
                )
                pending_setup = None
                if open_trade is not None:
                    continue

            if open_trade is None and pending_setup is not None:
                pending_setup["bars_evaluated"] = int(pending_setup["bars_evaluated"]) + 1
                if _confirmation_triggered(
                    row=row,
                    direction=int(pending_setup["direction"]),
                    reference_close=float(pending_setup["reference_close"]),
                    reference_range=float(pending_setup["reference_range"]),
                    pullback_fraction=float(pending_setup["pullback_fraction"]),
                ):
                    pending_setup["confirmed_for_next_open"] = idx < last_idx
                    if idx == last_idx:
                        pending_setup = None
                elif int(pending_setup["bars_evaluated"]) >= int(pending_setup["confirmation_window"]):
                    pending_setup = None

            if open_trade is not None or pending_setup is not None:
                continue

            signal = int(_float_or_nan(row.get("signal")))
            if signal == 0:
                continue

            direction = int(signal)
            if variant.entry_mode == "next_open":
                open_trade = _build_open_trade(
                    row=row,
                    direction=direction,
                    entry_price=_entry_market_fill(execution_model, open_, direction),
                    variant=variant,
                    instrument=instrument,
                    position_sizing=position_sizing,
                    capital_before_trade_usd=capital_before_trade_usd,
                    sizing_decisions=sizing_decisions,
                )
                continue

            reference_close = _float_or_nan(row.get("setup_reference_close"))
            reference_range = _float_or_nan(row.get("setup_reference_range"))

            if variant.entry_mode == "pullback_limit":
                if variant.pullback_fraction is None:
                    continue
                limit_fill = _maybe_fill_pullback_limit(
                    row=row,
                    direction=direction,
                    reference_close=reference_close,
                    reference_range=reference_range,
                    pullback_fraction=float(variant.pullback_fraction),
                )
                if limit_fill is None:
                    continue
                open_trade = _build_open_trade(
                    row=row,
                    direction=direction,
                    entry_price=float(limit_fill),
                    variant=variant,
                    instrument=instrument,
                    position_sizing=position_sizing,
                    capital_before_trade_usd=capital_before_trade_usd,
                    sizing_decisions=sizing_decisions,
                )
                continue

            if variant.entry_mode == "confirmation":
                if variant.pullback_fraction is None or variant.confirmation_window is None:
                    continue
                pending_setup = {
                    "direction": direction,
                    "reference_close": reference_close,
                    "reference_range": reference_range,
                    "pullback_fraction": float(variant.pullback_fraction),
                    "confirmation_window": int(variant.confirmation_window),
                    "bars_evaluated": 1,
                    "confirmed_for_next_open": False,
                    "setup_stop_reference_long": row.get("setup_stop_reference_long"),
                    "setup_stop_reference_short": row.get("setup_stop_reference_short"),
                    "setup_reference_atr": row.get("setup_reference_atr"),
                    "setup_reference_vwap": row.get("setup_reference_vwap"),
                    "setup_signal_time": row.get("setup_signal_time"),
                }
                if _confirmation_triggered(
                    row=row,
                    direction=direction,
                    reference_close=reference_close,
                    reference_range=reference_range,
                    pullback_fraction=float(variant.pullback_fraction),
                ):
                    pending_setup["confirmed_for_next_open"] = idx < last_idx
                    if idx == last_idx:
                        pending_setup = None
                elif int(pending_setup["bars_evaluated"]) >= int(pending_setup["confirmation_window"]):
                    pending_setup = None
                continue

            raise ValueError(f"Unsupported entry_mode '{variant.entry_mode}'.")

    trades_df = pd.DataFrame(trades) if trades else empty_trade_log()
    sizing_df = pd.DataFrame(sizing_decisions) if sizing_decisions else _empty_sizing_decision_log()
    if "variant_name" not in trades_df.columns:
        trades_df["variant_name"] = pd.Series(dtype=object)
    if "entry_signal_time" not in trades_df.columns:
        trades_df["entry_signal_time"] = pd.Series(dtype="datetime64[ns]")
    if "bars_held" not in trades_df.columns:
        trades_df["bars_held"] = pd.Series(dtype=float)
    for column in (
        "position_sizing_mode",
        "risk_pct_decimal",
        "stop_distance_points",
        "contracts_raw",
        "max_contracts",
        "skip_trade_if_too_small",
        "capital_after_trade_usd",
        "initial_stop_price",
    ):
        if column not in trades_df.columns:
            trades_df[column] = pd.Series(dtype=float if column not in {"position_sizing_mode", "skip_trade_if_too_small"} else object)
    return VolumeClimaxPullbackV2BacktestResult(trades=trades_df, sizing_decisions=sizing_df)


In [ ]:
SYMBOL = "MNQ"
INITIAL_CAPITAL_USD = 50_000.0
IS_FRACTION = 0.70
PLOT_TEMPLATE = "plotly_dark"

MNQ_1M_DATASET_PATH = None

PULLBACK_VOLUME_QUANTILE = 0.95
PULLBACK_VOLUME_LOOKBACK = 50
PULLBACK_MIN_BODY_FRACTION = 0.50
PULLBACK_MIN_RANGE_ATR = 1.20
PULLBACK_ENTRY_MODE = "next_open"
PULLBACK_PULLBACK_FRACTION = None
PULLBACK_CONFIRMATION_WINDOW = None
PULLBACK_EXIT_MODE = "atr_fraction"
PULLBACK_RR_TARGET = 1.0
PULLBACK_ATR_TARGET_MULTIPLE = 1.0
PULLBACK_TIME_STOP_BARS = 2
PULLBACK_TRAILING_ATR_MULTIPLE = 0.50

PULLBACK_RISK_PCT = 0.0025
PULLBACK_MAX_CONTRACTS = 6
PULLBACK_SKIP_TRADE_IF_TOO_SMALL = True
PULLBACK_COMPOUND_REALIZED_PNL = False

PULLBACK_VOLUME_QUANTILE_GRID = (0.90, 0.95, 0.975, 0.99)
PULLBACK_VOLUME_LOOKBACK_GRID = (30, 50, 70)
PULLBACK_MIN_BODY_FRACTION_GRID = (0.40, 0.50, 0.60, 0.70)
PULLBACK_MIN_RANGE_ATR_GRID = (1.0, 1.2, 1.5, 1.8)
PULLBACK_ENTRY_LABEL_GRID = (
    ("next_open", None, None),
    ("pullback_limit", 0.25, None),
    ("pullback_limit", 0.50, None),
    ("confirmation", 0.25, 1),
    ("confirmation", 0.25, 2),
    ("confirmation", 0.50, 2),
)
PULLBACK_ATR_TARGET_GRID = (0.75, 1.0, 1.25, 1.5)
PULLBACK_RR_TARGET_GRID = (0.75, 1.0, 1.25, 1.5)
PULLBACK_TIME_STOP_BARS_GRID = (1, 2, 3, 4)
PULLBACK_TRAILING_ATR_MULTIPLE_GRID = (0.25, 0.50, 0.75, 1.0)
PULLBACK_RISK_PCT_GRID = (0.0015, 0.0020, 0.0025, 0.0030, 0.0035, 0.0040)
PULLBACK_MAX_CONTRACTS_GRID = (2, 3, 4, 5, 6)
PULLBACK_SKIP_TOO_SMALL_GRID = (False, True)
PULLBACK_COMPOUND_REALIZED_PNL_GRID = (False, True)

dataset_path = resolve_dataset_path(MNQ_1M_DATASET_PATH, SYMBOL, timeframe="1m")
instrument_spec = get_instrument_spec(SYMBOL)

parameter_rows = [
    {"section": "global", "parameter": "SYMBOL", "value": SYMBOL, "meaning": "Contrat analyse."},
    {"section": "global", "parameter": "INITIAL_CAPITAL_USD", "value": INITIAL_CAPITAL_USD, "meaning": "Capital de reference."},
    {"section": "global", "parameter": "IS_FRACTION", "value": IS_FRACTION, "meaning": "Split chronologique IS/OOS."},
    {"section": "data", "parameter": "MNQ_1M_DATASET_PATH", "value": str(dataset_path), "meaning": "Source minute du notebook."},
    {"section": "alpha", "parameter": "PULLBACK_VOLUME_QUANTILE", "value": PULLBACK_VOLUME_QUANTILE, "meaning": "Seuil de volume historique du setup climax."},
    {"section": "alpha", "parameter": "PULLBACK_VOLUME_LOOKBACK", "value": PULLBACK_VOLUME_LOOKBACK, "meaning": "Fenetre historique du quantile de volume."},
    {"section": "alpha", "parameter": "PULLBACK_MIN_BODY_FRACTION", "value": PULLBACK_MIN_BODY_FRACTION, "meaning": "Qualite minimale du corps de la bougie de setup."},
    {"section": "alpha", "parameter": "PULLBACK_MIN_RANGE_ATR", "value": PULLBACK_MIN_RANGE_ATR, "meaning": "Range minimum du climax en ATR."},
    {"section": "entry", "parameter": "PULLBACK_ENTRY_MODE", "value": PULLBACK_ENTRY_MODE, "meaning": "Mode d'entree retenu dans la sleeve portefeuille."},
    {"section": "exit", "parameter": "PULLBACK_EXIT_MODE", "value": PULLBACK_EXIT_MODE, "meaning": "Mode de sortie retenu: atr_fraction."},
    {"section": "exit", "parameter": "PULLBACK_ATR_TARGET_MULTIPLE", "value": PULLBACK_ATR_TARGET_MULTIPLE, "meaning": "Distance du target en ATR."},
    {"section": "exit", "parameter": "PULLBACK_TIME_STOP_BARS", "value": PULLBACK_TIME_STOP_BARS, "meaning": "Nombre de barres max en position."},
    {"section": "exit", "parameter": "PULLBACK_TRAILING_ATR_MULTIPLE", "value": PULLBACK_TRAILING_ATR_MULTIPLE, "meaning": "Parametre du mode mixed."},
    {"section": "risk", "parameter": "PULLBACK_RISK_PCT", "value": PULLBACK_RISK_PCT, "meaning": "Risque par trade utilise par defaut dans le blend client."},
    {"section": "risk", "parameter": "PULLBACK_MAX_CONTRACTS", "value": PULLBACK_MAX_CONTRACTS, "meaning": "Cap de contrats par trade."},
    {"section": "risk", "parameter": "PULLBACK_SKIP_TRADE_IF_TOO_SMALL", "value": PULLBACK_SKIP_TRADE_IF_TOO_SMALL, "meaning": "Skip si le sizing tombe sous 1 contrat."},
    {"section": "risk", "parameter": "PULLBACK_COMPOUND_REALIZED_PNL", "value": PULLBACK_COMPOUND_REALIZED_PNL, "meaning": "False = sizing sur capital initial constant."},
]
display(Markdown("## 0. Parametrage client"))
params = parameter_table(parameter_rows)


In [ ]:
raw_1m = load_symbol_data(SYMBOL, input_paths={SYMBOL: dataset_path})
raw_1m["timestamp"] = pd.to_datetime(raw_1m["timestamp"], errors="coerce")
bars_1h = resample_rth_1h(raw_1m)
bars_1h["timestamp"] = pd.to_datetime(bars_1h["timestamp"], errors="coerce")
bars_1h["session_date"] = pd.to_datetime(bars_1h["timestamp"]).dt.date
all_sessions = normalize_sessions(sorted(pd.to_datetime(bars_1h["session_date"]).dt.date.unique()))
is_sessions_raw, oos_sessions_raw = split_sessions(bars_1h[["session_date"]].copy(), ratio=IS_FRACTION)
is_sessions = normalize_sessions(is_sessions_raw)
oos_sessions = normalize_sessions(oos_sessions_raw)
features_1h = prepare_volume_climax_pullback_v2_features(bars_1h)

display(Markdown("## 1. Data et preparation 1h"))
display(pd.DataFrame(
    [
        {"item": "rows_1m", "value": f"{len(raw_1m):,}"},
        {"item": "rows_1h_rth", "value": f"{len(bars_1h):,}"},
        {"item": "sessions_all", "value": len(all_sessions)},
        {"item": "sessions_is", "value": len(is_sessions)},
        {"item": "sessions_oos", "value": len(oos_sessions)},
        {"item": "first_timestamp_1h", "value": str(bars_1h["timestamp"].min())},
        {"item": "last_timestamp_1h", "value": str(bars_1h["timestamp"].max())},
    ]
))

def make_pullback_variant(**overrides):
    payload = {
        "name": "client_pullback_retained",
        "family": "retained",
        "timeframe": "1h",
        "volume_quantile": PULLBACK_VOLUME_QUANTILE,
        "volume_lookback": PULLBACK_VOLUME_LOOKBACK,
        "min_body_fraction": PULLBACK_MIN_BODY_FRACTION,
        "min_range_atr": PULLBACK_MIN_RANGE_ATR,
        "trend_ema_window": None,
        "ema_slope_threshold": None,
        "atr_percentile_low": None,
        "atr_percentile_high": None,
        "compression_ratio_max": None,
        "entry_mode": PULLBACK_ENTRY_MODE,
        "pullback_fraction": PULLBACK_PULLBACK_FRACTION,
        "confirmation_window": PULLBACK_CONFIRMATION_WINDOW,
        "exit_mode": PULLBACK_EXIT_MODE,
        "rr_target": PULLBACK_RR_TARGET,
        "atr_target_multiple": PULLBACK_ATR_TARGET_MULTIPLE,
        "time_stop_bars": PULLBACK_TIME_STOP_BARS,
        "trailing_atr_multiple": PULLBACK_TRAILING_ATR_MULTIPLE,
        "session_overlay": "all_rth",
    }
    payload.update(overrides)
    return VolumeClimaxPullbackV2Variant(**payload)

def make_pullback_sizing(**overrides):
    payload = {
        "initial_capital_usd": INITIAL_CAPITAL_USD,
        "risk_pct": PULLBACK_RISK_PCT,
        "max_contracts": PULLBACK_MAX_CONTRACTS,
        "skip_trade_if_too_small": PULLBACK_SKIP_TRADE_IF_TOO_SMALL,
        "compound_realized_pnl": PULLBACK_COMPOUND_REALIZED_PNL,
    }
    payload.update(overrides)
    return RiskPercentPositionSizing(**payload)

def build_pullback_signal(variant):
    return build_volume_climax_pullback_v2_signal_frame(features_1h, variant)

def run_pullback_variant(variant, sizing, signal_df=None):
    signal_df = build_pullback_signal(variant) if signal_df is None else signal_df
    execution_model, instrument = build_execution_model_for_profile(symbol=SYMBOL, profile_name="repo_realistic")
    result = run_volume_climax_pullback_v2_backtest(
        signal_df=signal_df,
        variant=variant,
        execution_model=execution_model,
        instrument=instrument,
        position_sizing=sizing,
    )
    full_curve = daily_results_from_trades(result.trades, all_sessions, INITIAL_CAPITAL_USD)
    summary = summarize_strategy_scopes("Pullback retained", full_curve, result.trades, is_sessions, oos_sessions, INITIAL_CAPITAL_USD)
    return signal_df, result, full_curve, summary

retained_variant = make_pullback_variant()
retained_sizing = make_pullback_sizing()
pullback_signal_df, pullback_result, pullback_daily_full, pullback_summary = run_pullback_variant(retained_variant, retained_sizing)
pullback_trades = pullback_result.trades.copy()

display(Markdown("## 2. Reconstruction de la sleeve retenue"))
display(Markdown(
    "Par defaut, ce notebook reproduit l'alpha `dynamic_exit_atr_target_1p0_ts2_vq0p95_bf0p5_ra1p2` "
    "avec le sizing `risk_pct_0p0025__max_contracts_6__skip_trade_if_too_small_true`, c'est-a-dire la sleeve "
    "utilisee dans le notebook equal-weight client."
))
display(pd.DataFrame([asdict(retained_variant)]).T.rename(columns={0: "value"}))
display(pd.DataFrame([{
    "risk_pct": retained_sizing.risk_pct,
    "max_contracts": retained_sizing.max_contracts,
    "skip_trade_if_too_small": retained_sizing.skip_trade_if_too_small,
    "compound_realized_pnl": retained_sizing.compound_realized_pnl,
}]))
display(pullback_summary.round(3))

pullback_oos_curve = daily_results_from_trades(subset_trades(pullback_trades, oos_sessions), oos_sessions, INITIAL_CAPITAL_USD)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Full sample", "OOS rebased"))
fig.add_trace(go.Scatter(x=pullback_daily_full["session_date"], y=pullback_daily_full["equity"], mode="lines", name="Pullback full", line=dict(color="#16a34a", width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=pullback_oos_curve["session_date"], y=pullback_oos_curve["equity"], mode="lines", name="Pullback OOS", line=dict(color="#f59e0b", width=2.5)), row=1, col=2)
fig.update_layout(template=PLOT_TEMPLATE, width=1350, height=500, legend=dict(orientation="h", y=-0.15))
fig.update_yaxes(title_text="Equity USD", row=1, col=1)
fig.update_yaxes(title_text="Equity USD", row=1, col=2)
fig.show()

display(Markdown("### Extrait des trades"))
display(pullback_trades.head(20))


In [ ]:
display(Markdown("## 3. Heatmaps IS/OOS - Alpha pur"))

alpha_rows = []
for volume_quantile in PULLBACK_VOLUME_QUANTILE_GRID:
    for body_fraction in PULLBACK_MIN_BODY_FRACTION_GRID:
        for range_atr in PULLBACK_MIN_RANGE_ATR_GRID:
            variant = make_pullback_variant(
                volume_quantile=float(volume_quantile),
                min_body_fraction=float(body_fraction),
                min_range_atr=float(range_atr),
            )
            signal_df, result, full_curve, summary = run_pullback_variant(variant, retained_sizing)
            for _, row in summary.iterrows():
                alpha_rows.append(
                    {
                        "scope": row["scope"],
                        "volume_quantile": float(volume_quantile),
                        "min_body_fraction": float(body_fraction),
                        "min_range_atr": float(range_atr),
                        "net_pnl_usd": float(row["net_pnl_usd"]),
                        "sharpe": float(row["sharpe"]),
                        "max_drawdown_usd": float(row["max_drawdown_usd"]),
                    }
                )
pullback_alpha_grid = pd.DataFrame(alpha_rows)
for range_atr in sorted(pullback_alpha_grid["min_range_atr"].unique()):
    alpha_slice = pullback_alpha_grid.loc[pullback_alpha_grid["min_range_atr"].eq(range_atr)].copy()
    plot_is_oos_heatmaps(alpha_slice, "volume_quantile", "min_body_fraction", "sharpe", f"Pullback | volume quantile x body fraction | Sharpe | range_atr={range_atr}")
    plot_is_oos_heatmaps(alpha_slice, "volume_quantile", "min_body_fraction", "net_pnl_usd", f"Pullback | volume quantile x body fraction | Net PnL | range_atr={range_atr}", text_auto=".0f")

lookback_rows = []
for volume_lookback in PULLBACK_VOLUME_LOOKBACK_GRID:
    variant = make_pullback_variant(volume_lookback=int(volume_lookback))
    signal_df, result, full_curve, summary = run_pullback_variant(variant, retained_sizing)
    for _, row in summary.iterrows():
        lookback_rows.append(
            {
                "scope": row["scope"],
                "volume_lookback": int(volume_lookback),
                "net_pnl_usd": float(row["net_pnl_usd"]),
                "sharpe": float(row["sharpe"]),
                "max_drawdown_usd": float(row["max_drawdown_usd"]),
            }
        )
pullback_lookback_grid = pd.DataFrame(lookback_rows)
plot_scope_heatmap(pullback_lookback_grid, "volume_lookback", "sharpe", "Pullback | volume lookback | Sharpe")


In [ ]:
display(Markdown("## 4. Heatmaps IS/OOS - Modes d'entree et de sortie"))

entry_rows = []
for entry_mode, pullback_fraction, confirmation_window in PULLBACK_ENTRY_LABEL_GRID:
    variant = make_pullback_variant(
        entry_mode=str(entry_mode),
        pullback_fraction=pullback_fraction,
        confirmation_window=confirmation_window,
    )
    label = f"{entry_mode}|pf={pullback_fraction}|cw={confirmation_window}"
    signal_df, result, full_curve, summary = run_pullback_variant(variant, retained_sizing)
    for _, row in summary.iterrows():
        entry_rows.append(
            {
                "scope": row["scope"],
                "entry_label": label,
                "net_pnl_usd": float(row["net_pnl_usd"]),
                "sharpe": float(row["sharpe"]),
                "max_drawdown_usd": float(row["max_drawdown_usd"]),
            }
        )
pullback_entry_grid = pd.DataFrame(entry_rows)
plot_scope_heatmap(pullback_entry_grid, "entry_label", "sharpe", "Pullback | entry-mode family | Sharpe")

atr_exit_rows = []
for atr_target in PULLBACK_ATR_TARGET_GRID:
    for time_stop in PULLBACK_TIME_STOP_BARS_GRID:
        variant = make_pullback_variant(
            exit_mode="atr_fraction",
            atr_target_multiple=float(atr_target),
            time_stop_bars=int(time_stop),
        )
        signal_df, result, full_curve, summary = run_pullback_variant(variant, retained_sizing)
        for _, row in summary.iterrows():
            atr_exit_rows.append(
                {
                    "scope": row["scope"],
                    "atr_target_multiple": float(atr_target),
                    "time_stop_bars": int(time_stop),
                    "net_pnl_usd": float(row["net_pnl_usd"]),
                    "sharpe": float(row["sharpe"]),
                    "max_drawdown_usd": float(row["max_drawdown_usd"]),
                }
            )
pullback_atr_exit_grid = pd.DataFrame(atr_exit_rows)
plot_is_oos_heatmaps(pullback_atr_exit_grid, "atr_target_multiple", "time_stop_bars", "sharpe", "Pullback | atr target x time stop | Sharpe")

rr_exit_rows = []
for rr_target in PULLBACK_RR_TARGET_GRID:
    for time_stop in PULLBACK_TIME_STOP_BARS_GRID:
        variant = make_pullback_variant(
            exit_mode="fixed_rr",
            rr_target=float(rr_target),
            time_stop_bars=int(time_stop),
            atr_target_multiple=None,
        )
        signal_df, result, full_curve, summary = run_pullback_variant(variant, retained_sizing)
        for _, row in summary.iterrows():
            rr_exit_rows.append(
                {
                    "scope": row["scope"],
                    "rr_target": float(rr_target),
                    "time_stop_bars": int(time_stop),
                    "net_pnl_usd": float(row["net_pnl_usd"]),
                    "sharpe": float(row["sharpe"]),
                    "max_drawdown_usd": float(row["max_drawdown_usd"]),
                }
            )
pullback_rr_exit_grid = pd.DataFrame(rr_exit_rows)
plot_is_oos_heatmaps(pullback_rr_exit_grid, "rr_target", "time_stop_bars", "sharpe", "Pullback | fixed RR target x time stop | Sharpe")

mixed_rows = []
for trailing_atr in PULLBACK_TRAILING_ATR_MULTIPLE_GRID:
    for time_stop in PULLBACK_TIME_STOP_BARS_GRID:
        variant = make_pullback_variant(
            exit_mode="mixed",
            atr_target_multiple=None,
            trailing_atr_multiple=float(trailing_atr),
            time_stop_bars=int(time_stop),
        )
        signal_df, result, full_curve, summary = run_pullback_variant(variant, retained_sizing)
        for _, row in summary.iterrows():
            mixed_rows.append(
                {
                    "scope": row["scope"],
                    "trailing_atr_multiple": float(trailing_atr),
                    "time_stop_bars": int(time_stop),
                    "net_pnl_usd": float(row["net_pnl_usd"]),
                    "sharpe": float(row["sharpe"]),
                    "max_drawdown_usd": float(row["max_drawdown_usd"]),
                }
            )
pullback_mixed_grid = pd.DataFrame(mixed_rows)
plot_is_oos_heatmaps(pullback_mixed_grid, "trailing_atr_multiple", "time_stop_bars", "sharpe", "Pullback | mixed trailing ATR x time stop | Sharpe")


In [ ]:
display(Markdown("## 5. Heatmaps IS/OOS - Sizing et compounding"))

sizing_rows = []
for risk_pct in PULLBACK_RISK_PCT_GRID:
    for max_contracts in PULLBACK_MAX_CONTRACTS_GRID:
        sizing = make_pullback_sizing(risk_pct=float(risk_pct), max_contracts=int(max_contracts))
        signal_df, result, full_curve, summary = run_pullback_variant(retained_variant, sizing, signal_df=pullback_signal_df)
        for _, row in summary.iterrows():
            sizing_rows.append(
                {
                    "scope": row["scope"],
                    "risk_pct": float(risk_pct),
                    "max_contracts": int(max_contracts),
                    "net_pnl_usd": float(row["net_pnl_usd"]),
                    "sharpe": float(row["sharpe"]),
                    "max_drawdown_usd": float(row["max_drawdown_usd"]),
                }
            )
pullback_sizing_grid = pd.DataFrame(sizing_rows)
plot_is_oos_heatmaps(pullback_sizing_grid, "max_contracts", "risk_pct", "sharpe", "Pullback | risk pct x max contracts | Sharpe")
plot_is_oos_heatmaps(pullback_sizing_grid, "max_contracts", "risk_pct", "max_drawdown_usd", "Pullback | risk pct x max contracts | Max DD", text_auto=".0f", color_continuous_scale="RdYlGn_r")

bool_rows = []
for skip_small in PULLBACK_SKIP_TOO_SMALL_GRID:
    for compound_pnl in PULLBACK_COMPOUND_REALIZED_PNL_GRID:
        sizing = make_pullback_sizing(skip_trade_if_too_small=bool(skip_small), compound_realized_pnl=bool(compound_pnl))
        signal_df, result, full_curve, summary = run_pullback_variant(retained_variant, sizing, signal_df=pullback_signal_df)
        for _, row in summary.iterrows():
            bool_rows.append(
                {
                    "scope": row["scope"],
                    "skip_trade_if_too_small": str(bool(skip_small)),
                    "compound_realized_pnl": str(bool(compound_pnl)),
                    "net_pnl_usd": float(row["net_pnl_usd"]),
                    "sharpe": float(row["sharpe"]),
                    "max_drawdown_usd": float(row["max_drawdown_usd"]),
                }
            )
pullback_bool_grid = pd.DataFrame(bool_rows)
plot_is_oos_heatmaps(pullback_bool_grid, "compound_realized_pnl", "skip_trade_if_too_small", "sharpe", "Pullback | skip-small x compound-PnL | Sharpe")


In [ ]:
display(Markdown("## 6. Lecture finale"))
oos_row = pullback_summary.loc[pullback_summary["scope"].eq("oos")].iloc[0]
notes = [
    f"- Sleeve portefeuille par defaut: alpha `vq=0.95 / body=0.50 / range_atr=1.20 / atr_target=1.0 / time_stop=2`, sizing `{PULLBACK_RISK_PCT:.4f}` avec cap `{PULLBACK_MAX_CONTRACTS}` contrats.",
    f"- Pullback OOS du notebook par defaut: net `{fmt_money(oos_row['net_pnl_usd'])}`, Sharpe `{fmt_float(oos_row['sharpe'])}`, maxDD `{fmt_money(oos_row['max_drawdown_usd'])}`.",
    "- Le notebook separe bien alpha et sizing: on peut changer le setup sans toucher au moteur de risque, ou l'inverse.",
    "- Si l'objectif est la rigueur client, les premieres surfaces a lire sont : `volume_quantile / body_fraction / range_atr`, puis `entry-mode family`, puis `risk_pct / max_contracts`.",
]
display(Markdown("\n".join(notes)))
